In [ ]:
import requests

In [ ]:
url = r"https://webscraper.io/test-sites/e-commerce/static/computers/laptops"
response = requests.get(url)

In [ ]:
import re

image_urls = re.findall(
    r'https?://[^\s"\'<>]+\.(?:jpg|jpeg|png|webp)',
    response.text,
    re.IGNORECASE
)

for url in image_urls:
    print(url)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json

url = r"https://webscraper.io/test-sites/e-commerce/static/computers/laptops"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

product_cards = soup.select(".thumbnail")

products = []

for card in product_cards:
    # Product name
    name_tag = card.select_one(".title")
    name = (
        name_tag.get("title") or name_tag.get_text(strip=True)
        if name_tag else None
    )

    # Product price
    price_tag = card.select_one(".price")
    price = price_tag.get_text(strip=True) if price_tag else None

    # Product image
    image_tag = card.select_one("img")
    image = None

    if image_tag:
        image_path = (
            image_tag.get("src")
            or image_tag.get("data-src")
            or image_tag.get("data-original")
        )

        if image_path:
            image = urljoin(url, image_path)

    products.append({
        "name": name,
        "price": price,
        "image": image
    })

# Print JSON
json_data = json.dumps(
    products,
    indent=4,
    ensure_ascii=False
)

print(json_data)

# Save JSON file
with open("products.json", "w", encoding="utf-8") as file:
    file.write(json_data)

print(f"\nTotal products: {len(products)}")
print("JSON saved to products.json")

In [ ]:

import json
import csv

# Read JSON file
with open("products.json", "r", encoding="utf-8") as file:
    products = json.load(file)

# Convert JSON to CSV
with open("products.csv", "w", newline="", encoding="utf-8-sig") as file:
    fieldnames = ["name", "price", "image"]

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(products)

print("JSON successfully converted to CSV!")
print("File saved as products.csv")